# 04 — Design Feature Importance

Goal: identify which creative design attributes (scores, flags, dimensions) correlate with and predict performance. Uses both correlation analysis and a simple RandomForest classifier.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score

sns.set_theme(style='whitegrid', palette='muted')
DATA = Path('../')

cs = pd.read_csv(DATA / 'creative_summary.csv')
print(f'Shape: {cs.shape}')

## 1. Correlation Heatmap: Design Features vs KPIs

In [ ]:
design_features = [
    'readability_score', 'brand_visibility_score', 'clutter_score',
    'novelty_score', 'motion_score', 'text_density',
    'faces_count', 'product_count',
    'has_price', 'has_discount_badge', 'has_gameplay', 'has_ugc_style',
    'duration_sec'
]
kpi_targets = ['overall_ctr', 'overall_cvr', 'overall_roas', 'overall_ipm', 'perf_score']

corr_matrix = cs[design_features + kpi_targets].corr().loc[design_features, kpi_targets]

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr_matrix,
    annot=True, fmt='.2f', center=0,
    cmap='RdYlGn', linewidths=0.5, ax=ax,
    cbar_kws={'label': 'Pearson Correlation'}
)
ax.set_title('Design Features vs Performance KPIs — Correlation Heatmap', fontweight='bold', pad=12)
ax.set_xlabel('KPI')
ax.set_ylabel('Design Feature')
plt.tight_layout()
plt.show()

print('\nTop correlations with perf_score:')
print(corr_matrix['perf_score'].sort_values(key=abs, ascending=False).round(3))

## 2. Binary Flag Analysis — Has Price / Discount / Gameplay / UGC

In [ ]:
binary_flags = ['has_price', 'has_discount_badge', 'has_gameplay', 'has_ugc_style']
flag_labels = ['Has Price', 'Has Discount Badge', 'Has Gameplay', 'Has UGC Style']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for col_idx, (flag, label) in enumerate(zip(binary_flags, flag_labels)):
    for row_idx, (kpi, kpi_label) in enumerate([('overall_ctr', 'CTR'), ('overall_roas', 'ROAS')]):
        ax = axes[row_idx, col_idx]
        group_0 = cs[cs[flag] == 0][kpi].dropna()
        group_1 = cs[cs[flag] == 1][kpi].dropna()
        bp = ax.boxplot([group_0, group_1], labels=['No', 'Yes'], patch_artist=True, notch=False)
        bp['boxes'][0].set_facecolor('#AED6F1')
        bp['boxes'][1].set_facecolor('#A9DFBF')
        ax.set_title(f'{label}\nvs {kpi_label}', fontsize=8, fontweight='bold')
        ax.set_ylabel(kpi_label, fontsize=8)

plt.suptitle('Binary Creative Flags vs CTR and ROAS', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Static vs Video (duration_sec = 0 means static)

In [ ]:
cs['is_video'] = (cs['duration_sec'] > 0).map({True: 'Video', False: 'Static'})

fig, axes = plt.subplots(1, 3, figsize=(13, 5))
for ax, kpi, label in [
    (axes[0], 'overall_ctr',  'CTR'),
    (axes[1], 'overall_cvr',  'CVR'),
    (axes[2], 'overall_roas', 'ROAS')
]:
    data = [cs[cs['is_video'] == v][kpi].dropna() for v in ['Static', 'Video']]
    bp = ax.boxplot(data, labels=['Static', 'Video'], patch_artist=True, notch=False)
    bp['boxes'][0].set_facecolor('#AED6F1')
    bp['boxes'][1].set_facecolor('#F9E79F')
    ax.set_title(f'Static vs Video — {label}', fontweight='bold')
    ax.set_ylabel(label)

plt.suptitle('Static vs Video Creatives Performance Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(cs.groupby('is_video')[['overall_ctr','overall_cvr','overall_roas','perf_score']].median().round(4))

## 4. RandomForest Feature Importance (top_performer vs rest)

In [ ]:
# Label: 1 = top_performer, 0 = everything else
cs['is_top'] = (cs['creative_status'] == 'top_performer').astype(int)

# Encode categorical features
cat_features = ['vertical', 'format', 'theme', 'hook_type', 'dominant_color', 'emotional_tone', 'language']
X_cat = pd.get_dummies(cs[cat_features], drop_first=True)

X_num = cs[design_features].fillna(0)
X = pd.concat([X_num, X_cat], axis=1)
y = cs['is_top']

rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42, n_jobs=-1)
scores = cross_val_score(rf, X, y, cv=5, scoring='roc_auc')
print(f'RandomForest AUC (5-fold CV): {scores.mean():.3f} ± {scores.std():.3f}')

rf.fit(X, y)
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
top20_feat = importances.head(20)

fig, ax = plt.subplots(figsize=(10, 7))
top20_feat.sort_values().plot.barh(ax=ax, color='#4C72B0', edgecolor='white')
ax.set_title('Top 20 Feature Importances\n(RandomForest: top_performer vs rest)', fontweight='bold')
ax.set_xlabel('Mean Decrease in Impurity')
plt.tight_layout()
plt.show()

print('\nTop 10 features:')
print(top20_feat.head(10).round(4))

## 5. Novelty & Motion Score vs perf_score (violin plots)

In [ ]:
cs['novelty_bin'] = pd.cut(cs['novelty_score'], bins=4, labels=['Low','Med-Low','Med-High','High'])
cs['motion_bin']  = pd.cut(cs['motion_score'],  bins=4, labels=['Low','Med-Low','Med-High','High'])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, col, title in [
    (axes[0], 'novelty_bin', 'Novelty Score Quartile vs perf_score'),
    (axes[1], 'motion_bin',  'Motion Score Quartile vs perf_score')
]:
    sns.violinplot(data=cs, x=col, y='perf_score', ax=ax,
                   order=['Low','Med-Low','Med-High','High'],
                   palette='Blues', inner='quartile')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('perf_score')

plt.suptitle('Design Score Bins vs Overall Performance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Aspect Ratio Analysis

In [ ]:
cs['aspect_ratio'] = cs['width'] / cs['height']

def classify_ratio(r):
    if r < 0.6:   return 'Portrait (<0.6)'
    elif r < 0.9: return 'Tall (0.6–0.9)'
    elif r < 1.1: return 'Square (~1.0)'
    else:         return 'Landscape (>1.1)'

cs['ratio_class'] = cs['aspect_ratio'].apply(classify_ratio)

ratio_perf = cs.groupby('ratio_class')[['overall_ctr','overall_cvr','overall_roas','perf_score']].median()
print('Performance by aspect ratio class:')
print(ratio_perf.round(4))

fig, ax = plt.subplots(figsize=(9, 5))
ratio_perf['perf_score'].sort_values().plot.barh(ax=ax, color='#55A868', edgecolor='white')
ax.set_title('Median perf_score by Aspect Ratio Class', fontweight='bold')
ax.set_xlabel('Median perf_score')
plt.tight_layout()
plt.show()

## 7. Feature Impact Summary Table

In [ ]:
impact = corr_matrix['perf_score'].sort_values(key=abs, ascending=False)
impact_df = impact.reset_index()
impact_df.columns = ['feature', 'corr_with_perf_score']
impact_df['direction'] = impact_df['corr_with_perf_score'].apply(
    lambda x: 'positive' if x > 0.05 else ('negative' if x < -0.05 else 'neutral')
)
impact_df['strength'] = impact_df['corr_with_perf_score'].abs().apply(
    lambda x: 'strong' if x > 0.15 else ('moderate' if x > 0.07 else 'weak')
)
print(impact_df.to_string(index=False))